In [1]:

from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
import re
import os

import numpy as np
import pandas as pd

import scipy.signal as sp_signal
from scipy.signal import find_peaks, welch

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    confusion_matrix, classification_report,
 )
# Random seed global (utilisé dans ECG/Resp/Fusion)
RANDOM_STATE = 42

## Modèle ECG (suivant *tutoré ECG*)


In [2]:
# ===================== ECG: entraînement (style "tutoré ECG") =====================
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score

df_ecg_train = pd.read_csv("df_ecg_train.csv")
df_ecg_test  = pd.read_csv("df_ecg_test.csv")

ECG_TARGET_COL = "y"
ECG_ID_COLS = ["pair_id", "subject_id", "session_id", "file_path_ecg", ECG_TARGET_COL]

# Certaines versions de df_ecg_train/df_ecg_test contiennent déjà une colonne p_ecg (on l'ignore ici)
drop_extra = [c for c in ["p_ecg"] if c in df_ecg_train.columns]
FEATURE_COLS_ECG = [c for c in df_ecg_train.columns if c not in set(ECG_ID_COLS + drop_extra)]

X_train_ecg = df_ecg_train[FEATURE_COLS_ECG].copy()
y_train_ecg = df_ecg_train[ECG_TARGET_COL].astype(int).copy()
groups_train_ecg = df_ecg_train["subject_id"].astype(str).copy()

X_test_ecg = df_ecg_test[FEATURE_COLS_ECG].copy()
y_test_ecg = df_ecg_test[ECG_TARGET_COL].astype(int).copy()

print("=" * 60)
print("ECG DATA")
print("=" * 60)
print(f"TRAIN ECG: X={X_train_ecg.shape} | y={y_train_ecg.shape} | n_subjects={groups_train_ecg.nunique()}")
print(f"TEST  ECG: X={X_test_ecg.shape} | y={y_test_ecg.shape}")

# Pipeline + GridSearch AUC (comme tutoré ECG)
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", SVC(probability=True, random_state=RANDOM_STATE)),
])

param_grid = {
    "classifier__C": [0.1, 1, 10, 100],
    "classifier__kernel": ["rbf", "linear"],
    "classifier__gamma": ["scale", "auto"],
    "classifier__class_weight": ["balanced", None],
}

n_splits = min(5, int(groups_train_ecg.nunique()))
if n_splits < 2:
    raise ValueError("Pas assez de sujets ECG pour faire une GroupKFold (n_splits<2).")

gkf_ecg = GroupKFold(n_splits=n_splits)

grid_search_ecg = GridSearchCV(
    pipeline,
    param_grid,
    cv=gkf_ecg,
    scoring="roc_auc",
    n_jobs=-1,
    verbose=0,
    refit=True,
 )

print("🚀 ECG: tuning (AUC) via GroupKFold...")
grid_search_ecg.fit(X_train_ecg, y_train_ecg, groups=groups_train_ecg)

ecg_best_model = grid_search_ecg.best_estimator_
best_model_ecg = ecg_best_model  # alias utilisé par la partie fusion
print(f"✅ ECG best_params: {grid_search_ecg.best_params_}")
print(f"✅ ECG best_cv_auc: {grid_search_ecg.best_score_:.4f}")

# Probas (train/test)
p_ecg_train = ecg_best_model.predict_proba(X_train_ecg)[:, 1]
p_ecg_test  = ecg_best_model.predict_proba(X_test_ecg)[:, 1]

# Attache p_ecg aux tables (utilisé par la fusion)
df_ecg_train = df_ecg_train.copy()
df_ecg_test = df_ecg_test.copy()
df_ecg_train["p_ecg"] = p_ecg_train
df_ecg_test["p_ecg"] = p_ecg_test

# OOF probas (pour entraîner alpha sans fuite / sans utiliser le TEST)
from sklearn.base import clone
p_ecg_oof = np.zeros(len(y_train_ecg), dtype=float)
for tr_idx, va_idx in gkf_ecg.split(X_train_ecg, y_train_ecg, groups_train_ecg):
    est = clone(ecg_best_model)
    est.fit(X_train_ecg.iloc[tr_idx], y_train_ecg.iloc[tr_idx])
    p_ecg_oof[va_idx] = est.predict_proba(X_train_ecg.iloc[va_idx])[:, 1]

print(f"ECG AUC (OOF):  {roc_auc_score(y_train_ecg, p_ecg_oof):.4f}")
print(f"ECG AUC (TEST): {roc_auc_score(y_test_ecg, p_ecg_test):.4f}")

ECG DATA
TRAIN ECG: X=(100, 4) | y=(100,) | n_subjects=16
TEST  ECG: X=(26, 4) | y=(26,)
🚀 ECG: tuning (AUC) via GroupKFold...
✅ ECG best_params: {'classifier__C': 100, 'classifier__class_weight': 'balanced', 'classifier__gamma': 'scale', 'classifier__kernel': 'rbf'}
✅ ECG best_cv_auc: 0.7228
ECG AUC (OOF):  0.6890
ECG AUC (TEST): 0.6875


In [3]:
# 2) CM avec seuil optimisé sur TRAIN (OOF) si la fonction existe
if "best_threshold_balanced" in globals():
    best_ecg_thr = best_threshold_balanced(p_ecg_oof, y_train_ecg.values)
    thr_ecg_opt = best_ecg_thr["thr"]
    y_pred_ecg_opt = (np.asarray(p_ecg_test) >= thr_ecg_opt).astype(int)
    cm_ecg_opt = confusion_matrix(y_test_ecg, y_pred_ecg_opt)
    print("=" * 60)
    print(f"ECG CM (seuil optimisé OOF: {thr_ecg_opt:.2f})")
    print("=" * 60)
    print(cm_ecg_opt)
    print(f"Balanced Acc: {balanced_accuracy_score(y_test_ecg, y_pred_ecg_opt):.4f}")
    print(classification_report(y_test_ecg, y_pred_ecg_opt, target_names=['Sain','Diabétique'], zero_division=0))
else:
    print("best_threshold_balanced non défini (exécute d'abord la partie LOOCV respiration si tu veux le seuil optimisé).")

best_threshold_balanced non défini (exécute d'abord la partie LOOCV respiration si tu veux le seuil optimisé).


## Modèle respiration (Breathing) (suivant *tutoré breathing*)


In [4]:
train_df = pd.read_csv("features_train_df.csv")
test_df  = pd.read_csv("features_test_df.csv")

files= test_df["subject_id"].values
X_train = train_df.drop(columns=["y_pair","pair_id","subject_id","session_id","file_path_breath"])
y_train = train_df["y_pair"].values

X_test = test_df.drop(columns=["y_pair","pair_id","subject_id","session_id","file_path_breath"])
y_test = test_df["y_pair"].values

In [5]:
# Normalisation "standard" (recommandée) : fit sur TRAIN, transform sur TRAIN + TEST
# Objectif: éviter toute fuite d'information du TEST dans le pré-traitement.

from sklearn.preprocessing import MinMaxScaler

# Garder person (utile si tu veux plus tard une CV par personne)
train_person = train_df['subject_id'].copy()
test_person = test_df['subject_id'].copy()

# Séparer features et labels (TRAIN)
X_train_raw = train_df.drop(["y_pair","pair_id","subject_id","session_id","file_path_breath"], axis=1)
y_train = train_df['y_pair']

# Séparer features et labels (TEST)
X_test_raw = test_df.drop(["y_pair", "pair_id","subject_id","session_id","file_path_breath"], axis=1)
y_test = test_df['y_pair']

assert list(X_train_raw.columns) == list(X_test_raw.columns), "Colonnes train/test différentes !"

print("=" * 60)
print("NORMALISATION MinMaxScaler (0-1) - FIT sur TRAIN")
print("=" * 60)
print(f"TRAIN brut: {X_train_raw.shape} | TEST brut: {X_test_raw.shape}")

scaler = MinMaxScaler(feature_range=(0, 1))
X_train = pd.DataFrame(
    scaler.fit_transform(X_train_raw),
    columns=X_train_raw.columns,
    index=X_train_raw.index
 )
X_test = pd.DataFrame(
    scaler.transform(X_test_raw),
    columns=X_test_raw.columns,
    index=X_test_raw.index
 )





NORMALISATION MinMaxScaler (0-1) - FIT sur TRAIN
TRAIN brut: (100, 18) | TEST brut: (26, 18)


In [6]:
print("=" * 60)
print("ENTRAÎNEMENT DES MODÈLES (candidats + pondération par personne)")
print("=" * 60)

from sklearn.ensemble import (
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier,
    HistGradientBoostingClassifier,
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils.class_weight import compute_sample_weight

# Dictionnaire pour stocker les modèles
models = {}

# --- Pondération: éviter qu'une personne avec beaucoup de sessions domine ---
# (utile quand plusieurs sessions par personne)
train_person = train_person.reindex(y_train.index)

# Poids de classe (minorité => + lourd)
sample_w_class = compute_sample_weight(class_weight="balanced", y=y_train)

# Poids par personne: somme ~1 par personne (réduit l'effet "personnes très enregistrées")
per_person_counts = train_person.value_counts()
sample_w_person = train_person.map(lambda p: 1.0 / float(per_person_counts.loc[p])).values

# Combinaison (pour les modèles utilisant sample_weight)
sample_w = sample_w_class * sample_w_person
# Renormaliser pour garder une échelle stable
sample_w = sample_w / float(np.mean(sample_w))

print(f"Pondération: n_personnes_train={train_person.nunique()} | mean_w={sample_w.mean():.3f}")



# 2) Random Forest
print("2️⃣ Random Forest (class_weight=balanced_subsample + sample_weight par personne)...")
rf_model = RandomForestClassifier(
    n_estimators=1000,
    random_state=42,
    class_weight="balanced_subsample",
    min_samples_leaf=2,
    n_jobs=-1,
)
rf_model.fit(X_train, y_train, sample_weight=sample_w_person)
models["Random Forest"] = rf_model
print(f"   ✓ Entraîné sur shape: {X_train.shape}")






ENTRAÎNEMENT DES MODÈLES (candidats + pondération par personne)
Pondération: n_personnes_train=23 | mean_w=1.000
2️⃣ Random Forest (class_weight=balanced_subsample + sample_weight par personne)...
   ✓ Entraîné sur shape: (100, 18)


In [7]:
# ===================== LOOCV: choix du seuil (SANS TOP-4) =====================
# Ici on calcule un seuil par modèle sur TRAIN via Leave-One-Out CV,
# mais on NE filtre PAS les modèles: on garde tous les modèles candidats.

from sklearn.model_selection import LeaveOneOut
from sklearn.base import clone
from sklearn.metrics import balanced_accuracy_score, f1_score


def get_positive_proba(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        s = model.decision_function(X)
        return 1 / (1 + np.exp(-s))
    raise ValueError("Pas de score/proba disponible")


def make_cv_estimator(fitted_model):
    """Clone le modèle et réduit un peu les hyperparams coûteux pour rendre LOOCV faisable."""
    est = clone(fitted_model)

    try:
        if hasattr(est, "n_estimators"):
            est.set_params(n_estimators=min(int(getattr(est, "n_estimators")), 200))
    except Exception:
        pass

    try:
        if hasattr(est, "max_iter"):
            est.set_params(max_iter=min(int(getattr(est, "max_iter")), 200))
    except Exception:
        pass

    try:
        if hasattr(est, "n_jobs"):
            est.set_params(n_jobs=-1)
    except Exception:
        pass

    return est


def oof_positive_proba_loocv(fitted_model, X, y, sample_weight=None):
    loo = LeaveOneOut()
    oof = np.zeros(len(y), dtype=float)

    for train_idx, test_idx in loo.split(X, y):
        est = make_cv_estimator(fitted_model)

        X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
        X_te = X.iloc[test_idx]

        if sample_weight is not None:
            sw_tr = np.asarray(sample_weight)[train_idx]
            try:
                est.fit(X_tr, y_tr, sample_weight=sw_tr)
            except TypeError:
                est.fit(X_tr, y_tr)
        else:
            est.fit(X_tr, y_tr)

        oof[test_idx[0]] = float(get_positive_proba(est, X_te)[0])

    return oof


def best_threshold_balanced(proba, y):
    thresholds = np.linspace(0.05, 0.95, 91)
    best = {
        "thr": 0.5,
        "bal_acc": -1.0,
        "f1_macro": -1.0,
        "recall_diab": -1.0,
        "specificity_sain": -1.0,
    }

    y = np.asarray(y)

    for t in thresholds:
        y_hat = (proba >= t).astype(int)

        bal_acc = balanced_accuracy_score(y, y_hat)
        f1_macro = f1_score(y, y_hat, average="macro", zero_division=0)

        tp = int(((y == 1) & (y_hat == 1)).sum())
        fn = int(((y == 1) & (y_hat == 0)).sum())
        recall_diab = tp / (tp + fn) if (tp + fn) else 0.0

        tn = int(((y == 0) & (y_hat == 0)).sum())
        fp = int(((y == 0) & (y_hat == 1)).sum())
        specificity = tn / (tn + fp) if (tn + fp) else 0.0

        # Critère principal: balanced accuracy (bon pour les 2 classes)
        # Tie-breaker: f1_macro, puis seuil plus proche de 0.5
        if (
            (bal_acc > best["bal_acc"])
            or (bal_acc == best["bal_acc"] and f1_macro > best["f1_macro"])
            or (
                bal_acc == best["bal_acc"]
                and f1_macro == best["f1_macro"]
                and abs(t - 0.5) < abs(best["thr"] - 0.5)
            )
        ):
            best = {
                "thr": float(t),
                "bal_acc": float(bal_acc),
                "f1_macro": float(f1_macro),
                "recall_diab": float(recall_diab),
                "specificity_sain": float(specificity),
            }

    return best


# sample_w existe si tu as exécuté la cellule d'entraînement; sinon on le calcule ici
try:
    _ = sample_w
except NameError:
    from sklearn.utils.class_weight import compute_sample_weight

    sample_w = compute_sample_weight(class_weight="balanced", y=y_train)

print("=" * 60)
print("LOOCV: seuil optimal (Balanced Accuracy) — SANS TOP-4")
print("=" * 60)

thresholds_by_model = {}
cv_rows = []

for name, fitted_model in models.items():
    try:
        print(f"\n→ LOOCV sur: {name}")
        proba_oof = oof_positive_proba_loocv(fitted_model, X_train, y_train, sample_weight=sample_w)
        best = best_threshold_balanced(proba_oof, y_train.values)

        thresholds_by_model[name] = best
        cv_rows.append({"Model": name, **best})

        print(
            f"{name:>22} | thr={best['thr']:.2f} | bal_acc={best['bal_acc']:.3f} | f1_macro={best['f1_macro']:.3f} | "
            f"recall(diabet)={best['recall_diab']:.3f} | specificity(sain)={best['specificity_sain']:.3f}"
        )

    except Exception as e:
        thresholds_by_model[name] = {
            "thr": 0.5,
            "bal_acc": -1.0,
            "f1_macro": -1.0,
            "recall_diab": -1.0,
            "specificity_sain": -1.0,
        }
        cv_rows.append({"Model": name, **thresholds_by_model[name]})
        print(f"{name:>22} | seuil=0.50 (fallback) | raison: {e}")

cv_df = pd.DataFrame(cv_rows).sort_values(["bal_acc", "f1_macro"], ascending=False)
print("\n" + "=" * 60)
print("RÉSULTATS LOOCV (triés)")
print("=" * 60)
display(cv_df)

print("\n✓ LOOCV seuils terminés (aucun modèle supprimé)")


LOOCV: seuil optimal (Balanced Accuracy) — SANS TOP-4

→ LOOCV sur: Random Forest
         Random Forest | thr=0.48 | bal_acc=0.786 | f1_macro=0.800 | recall(diabet)=0.647 | specificity(sain)=0.924

RÉSULTATS LOOCV (triés)


,Model,thr,bal_acc,f1_macro,recall_diab,specificity_sain
0,Random Forest,0.48,0.785651,0.799505,0.647059,0.924242



✓ LOOCV seuils terminés (aucun modèle supprimé)


In [8]:
# ===================== ÉVALUATION (focus détection diabétique) =====================
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, fbeta_score, balanced_accuracy_score
  )

def eval_with_threshold(y_true, proba, thr):
    y_pred = (proba >= thr).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'f2': fbeta_score(y_true, y_pred, beta=2.0, zero_division=0),
        'specificity': specificity,
        'confusion_matrix': cm,
        'y_pred': y_pred
    }

print("=" * 60)
print("ÉVALUATION SUR TEST SET")
print("=" * 60)
print(f"Test set shape: X_test={X_test.shape}, y_test={y_test.shape}")

results = {}

for model_name, model in models.items():
    # Probas/score
    if hasattr(model, 'predict_proba'):
        proba_test = model.predict_proba(X_test)[:, 1]
    else:
        s = model.decision_function(X_test)
        proba_test = 1 / (1 + np.exp(-s))

    thr = thresholds_by_model.get(model_name, {}).get('thr', 0.5)
    m = eval_with_threshold(y_test.values, proba_test, thr)
    results[model_name] = m

    print(f"\n{'='*60}")
    print(f"📊 {model_name.upper()} | seuil={thr:.2f}")
    print(f"{'='*60}")
    print(f"Accuracy:          {m['accuracy']:.4f}")
    print(f"Balanced Accuracy: {m['balanced_accuracy']:.4f}")
    print(f"Precision (diab):  {m['precision']:.4f}")
    print(f"Recall (diab):     {m['recall']:.4f}  ← sensibilité")
    print(f"Specificity (sain):{m['specificity']:.4f}")
    print(f"F1:                {m['f1']:.4f}")
    print(f"F2 (focus recall): {m['f2']:.4f}")

    cm = m['confusion_matrix']
    print(f"\nConfusion Matrix:")
    print(f"                Prédits")
    print(f"                Sain    Diabétique")
    print(f"Réels Sain      {cm[0,0]:<7} {cm[0,1]}")
    print(f"      Diabétique {cm[1,0]:<7} {cm[1,1]}")

    print("\nClassification Report:")
    print(classification_report(y_test, m['y_pred'], target_names=['Sain', 'Diabétique'], zero_division=0))

print("\n✓ Évaluation complétée")

ÉVALUATION SUR TEST SET
Test set shape: X_test=(26, 18), y_test=(26,)

📊 RANDOM FOREST | seuil=0.48
Accuracy:          0.7692
Balanced Accuracy: 0.7562
Precision (diab):  0.7000
Recall (diab):     0.7000  ← sensibilité
Specificity (sain):0.8125
F1:                0.7000
F2 (focus recall): 0.7000

Confusion Matrix:
                Prédits
                Sain    Diabétique
Réels Sain      13      3
      Diabétique 3       7

Classification Report:
              precision    recall  f1-score   support

        Sain       0.81      0.81      0.81        16
  Diabétique       0.70      0.70      0.70        10

    accuracy                           0.77        26
   macro avg       0.76      0.76      0.76        26
weighted avg       0.77      0.77      0.77        26


✓ Évaluation complétée


In [9]:
# ===============================
# FUSION ECG + Respiration avec Alpha optimisé (OOF)
# ===============================
from sklearn.model_selection import GroupKFold
from sklearn.base import clone
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (recall_score, fbeta_score, balanced_accuracy_score,
                             roc_auc_score, accuracy_score, confusion_matrix, classification_report)
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42

def _specificity(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    return tn / (tn + fp) if (tn + fp) else 0.0

# --- Préparation données ---
# ECG OOF déjà calculé (p_ecg_oof)
oof_ecg_s = pd.Series(p_ecg_oof, index=df_ecg_train['pair_id']).dropna()
y_ecg_s = pd.Series(y_train_ecg.values, index=df_ecg_train['pair_id'])

# Respiration OOF (GroupKFold)
FEATURE_COLS_RESP = X_train_raw.columns.tolist()
g_resp = train_df['subject_id'].copy()
pair_resp = train_df['pair_id'].copy()
y_resp = train_df['y_pair'].astype(int).copy()

n_splits_resp = min(5, int(g_resp.nunique()))
rf_params = {'n_estimators': 200, 'random_state': RANDOM_STATE, 'class_weight': 'balanced_subsample', 
             'min_samples_leaf': 2, 'n_jobs': -1}

oof_resp = np.full(len(train_df), np.nan, dtype=float)
gkf_resp = GroupKFold(n_splits=n_splits_resp)
for tr_idx, va_idx in gkf_resp.split(X_train_raw, y_resp, groups=g_resp):
    sc = MinMaxScaler()
    X_tr, X_va = sc.fit_transform(X_train_raw.iloc[tr_idx]), sc.transform(X_train_raw.iloc[va_idx])
    sw = g_resp.iloc[tr_idx].map(lambda p: 1.0 / g_resp.iloc[tr_idx].value_counts().loc[p]).values
    rf = RandomForestClassifier(**rf_params)
    rf.fit(X_tr, y_resp.iloc[tr_idx], sample_weight=sw)
    oof_resp[va_idx] = rf.predict_proba(X_va)[:, 1]

oof_resp_s = pd.Series(oof_resp, index=pair_resp).dropna()
y_resp_s = pd.Series(y_resp.values, index=pair_resp)

# --- Fusion OOF pour optimiser alpha ---
alpha_df = pd.DataFrame({'p_ecg': oof_ecg_s}).join(pd.DataFrame({'p_resp': oof_resp_s}), how='inner')
alpha_df['y_ecg'] = y_ecg_s.reindex(alpha_df.index)
alpha_df['y_resp'] = y_resp_s.reindex(alpha_df.index)
alpha_df['y'] = alpha_df['y_ecg'].fillna(alpha_df['y_resp']).astype(int)

# --- Grid search alpha (maximise recall diabétique) ---
alphas = np.linspace(0.1, 1.0, 1000)
best = {'alpha': 0.5, 'recall': -1, 'f2': -1, 'bal_acc': -1}
y_true = alpha_df['y'].values

for a in alphas:
    p_f = a * alpha_df['p_ecg'].values + (1.0 - a) * alpha_df['p_resp'].values
    y_hat = (p_f >= 0.5).astype(int)
    rec = recall_score(y_true, y_hat, pos_label=1, zero_division=0)
    f2 = fbeta_score(y_true, y_hat, beta=2.0, zero_division=0)
    bal = balanced_accuracy_score(y_true, y_hat)
    if rec > best['recall'] or (rec == best['recall'] and f2 > best['f2']):
        best = {'alpha': float(a), 'recall': rec, 'f2': f2, 'bal_acc': bal}

alpha_opt = best['alpha']
print("=" * 60)
print("ALPHA OPTIMISATION (TRAIN OOF)")
print("=" * 60)
print(f"Pairs utilisées: {len(alpha_df)}")
print(f"alpha_opt = {alpha_opt:.2f} | recall = {best['recall']:.3f} | F2 = {best['f2']:.3f} | bal_acc = {best['bal_acc']:.3f}")

# ===============================
# LATE FUSION SUR TEST
# ===============================
# Préparer p_resp sur test
rf_final = RandomForestClassifier(**rf_params)
sw_train = g_resp.map(lambda p: 1.0 / g_resp.value_counts().loc[p]).values
rf_final.fit(scaler.fit_transform(X_train_raw), y_resp, sample_weight=sw_train)
p_resp_test = rf_final.predict_proba(scaler.transform(X_test_raw))[:, 1]

# Fusion dataframe
fusion_df = pd.DataFrame({
    'pair_id': df_ecg_test['pair_id'],
    'y': df_ecg_test['y'],
    'p_ecg': p_ecg_test
}).merge(
    pd.DataFrame({'pair_id': test_df['pair_id'], 'p_resp': p_resp_test}),
    on='pair_id', how='inner'
)

fusion_df['p_fused'] = alpha_opt * fusion_df['p_ecg'] + (1.0 - alpha_opt) * fusion_df['p_resp']
fusion_df['y_pred'] = (fusion_df['p_fused'] >= 0.5).astype(int)

print("\n" + "=" * 60)
print("FUSION TEST RESULTS")
print("=" * 60)
print(f"Fusion rows: {len(fusion_df)} | alpha: {alpha_opt:.2f}")
try:
    print(f"ROC AUC: {roc_auc_score(fusion_df['y'], fusion_df['p_fused']):.4f}")
except: pass
print(f"Accuracy: {accuracy_score(fusion_df['y'], fusion_df['y_pred']):.4f}")
print(f"Recall: {recall_score(fusion_df['y'], fusion_df['y_pred']):.4f}")
print(f"Specificity: {_specificity(fusion_df['y'], fusion_df['y_pred']):.4f}")
print(f"\nConfusion Matrix:\n{confusion_matrix(fusion_df['y'], fusion_df['y_pred'])}")
print(f"\n{classification_report(fusion_df['y'], fusion_df['y_pred'], target_names=['Sain', 'Diabétique'])}")


ALPHA OPTIMISATION (TRAIN OOF)
Pairs utilisées: 100
alpha_opt = 0.10 | recall = 0.378 | F2 = 0.400 | bal_acc = 0.586

FUSION TEST RESULTS
Fusion rows: 26 | alpha: 0.10
ROC AUC: 0.7437
Accuracy: 0.8077
Recall: 0.7000
Specificity: 0.8750

Confusion Matrix:
[[14  2]
 [ 3  7]]

              precision    recall  f1-score   support

        Sain       0.82      0.88      0.85        16
  Diabétique       0.78      0.70      0.74        10

    accuracy                           0.81        26
   macro avg       0.80      0.79      0.79        26
weighted avg       0.81      0.81      0.81        26

